In [1]:
import os

os.chdir('..')

In [19]:
import sys

import torch
from transformers import LlamaForCausalLM, LlamaTokenizer
from tqdm import tqdm
from peft import PeftModel, get_peft_model, LoraConfig
import json
import numpy as np
from utils.instructor_retrieval import perform_search, initialize_index
from datasets import load_dataset
from utils.prompter import Prompter
from utils.instructor_retrieval import perform_search, get_embeddings
import torch
import torch.nn as nn
import torch.nn.functional as F

# Prompter is a utility class to create a prompt for a given input
prompter = Prompter("alpaca")

# Load previously computed model performance matrix (if present). This file is
# produced by `performance_based_selection/generate_results.py` and saved as
# `model_performance.npy`. We load it once at import-time so downstream code can
# consult model performance scores when needed.
def load_results_matrix(path: str = "./performance_large/model_performance.npy"):
    if not os.path.exists(path):
        # Not an error: just return None so callers can fall back to defaults
        print(f"[info] results matrix not found at: {path}")
        return None
    try:
        arr = np.load(path, allow_pickle=True)
        # Coerce to ndarray for consistent handling (may be an object array)
        arr = np.array(arr, dtype=np.float32)
        print(f"[info] loaded results matrix from {path} with shape {arr.shape}")
        return arr
    except Exception as e:
        print(f"[warning] failed to load results matrix {path}: {e}")
        return None

# Expose loaded matrix as a module-level variable; callers can check for None.
results_matrix = load_results_matrix()

def load_base_model(model_name_or_path='meta-llama/Llama-2-7b-hf'):
    """
    Load the base model and tokenizer from a given model path.
    """
    tokenizer = LlamaTokenizer.from_pretrained(model_name_or_path)
    tokenizer.pad_token_id = 0
    tokenizer.padding_side = "left"

    base_model = LlamaForCausalLM.from_pretrained(
        model_name_or_path, torch_dtype=torch.float16
    )
    base_model.bfloat16()
    return base_model, tokenizer

def load_peft_model(lora_module_list, base_model):
    """
    Load and configure PEFT (Parameter-Efficient Fine-Tuning) adapters onto the base model.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    lora_lists = []
    for i, lora_model in enumerate(lora_module_list):
        if i == 0:
            peft_model = PeftModel.from_pretrained(base_model, lora_model, f"adapter{i}")
        else:
            peft_model.load_adapter(lora_model, f"adapter{i}")
        lora_lists.append(f"adapter{i}")

    peft_model.set_adapter(lora_lists)
    peft_model = peft_model.to(device)
    peft_model.eval()
    return peft_model

class BilinearFusionScorer(nn.Module):
    """
    Learns Wi, Wr and returns softmax weights over K adapters given a batch of inputs.
    """
    def __init__(self, A_init: torch.Tensor, top_k: int, temperature: float = 1.0):
        super().__init__()
        self.register_buffer("A", A_init.clone())       # (K, d_a)
        self.top_k = top_k
        self.tau = temperature

    @torch.no_grad()
    def set_adapter_embeddings(self, A_new: torch.Tensor):
        self.A = A_new.clone().to(self.A.device)

    def forward(self, I: torch.Tensor, exclude_idx: int = None):
        """
        I: (B, d_in) input embeddings
        Returns:
          probs: (B, K) softmax weights per sample
          logits: (B, K)
        """
        logits = I @ self.A.t()        # (B, K)

        # Instead of a softmax distribution, return a one-hot vector where the
        # highest (allowed) logit gets probability 1 and all others 0.
        # Use device/dtype safe tensor creation so we match `logits` dtype/device.
        argmax_idx = logits.argmax(dim=-1, keepdim=True)  # (B, 1)
        probs = logits.new_zeros(logits.size())
        probs.scatter_(1, argmax_idx, 1.0)
        return probs, logits

        old = int(argmax_idx[0, 0].item())
        mask_array = np.ones(results_matrix.shape[1], dtype=bool)
        if exclude_idx is not None:
            mask_array[exclude_idx] = False
        
        row = results_matrix[old]*mask_array
        maxpos = np.flatnonzero(row==row.max())
        sel = old if old in maxpos else int(maxpos[0])
        argmax_idx[0, 0] = torch.tensor(sel, device=argmax_idx.device)

        probs = logits.new_zeros(logits.size())
        probs.scatter_(1, argmax_idx, 1.0)
        return probs, logits
    
res_path = "./test_results/selection_new.json"
data_path = "./dataset/combined_test.json"
model_size = "7b"  # "7b" or "13b"
config_path = "./config/config2.json"
ood = False
batch_size = 1
correct_count = 0
results = []  # Initialize a list to store question and response data
device = "cuda" if torch.cuda.is_available() else "cpu"

cfg_path = "./performance_based_selection/models/base_model.config.json"
ckpt_path = "./performance_based_selection/models/base_model.pt"
    # Load scorer
with open(cfg_path) as f:
    cfg = json.load(f)

scorer = BilinearFusionScorer(
    A_init=torch.zeros(cfg["K"], 768, dtype=torch.bfloat16),  # placeholder, will be loaded
    top_k=3,
    temperature=cfg["temperature"],
).to(device)
if cfg["dtype"] == "bfloat16":
    scorer.bfloat16()

state = torch.load(ckpt_path, map_location="cpu")
scorer.load_state_dict(state)
scorer.to(device).eval()

model_outputs_folder = "./performance_large/outputs/"

model_output_dicts = [{} for _ in range(48)]

for file_name in os.listdir(model_outputs_folder):
    if not file_name.startswith("model_test"):
        continue

    model_index = int(file_name.split('_')[-1].split('.')[0])

    with open(os.path.join(model_outputs_folder, file_name), 'r', encoding='utf-8') as f:
        json_data = json.load(f)

        for sample in json_data:
            model_output_dicts[model_index][sample['inputs']] = sample['predicted_answer']


# Load the dataset
if data_path.endswith(".json") or data_path.endswith(".jsonl"):
    dataset = load_dataset("json", data_files=data_path)
else:
    dataset = load_dataset(data_path)

# Prepare the dataset with full prompts
eval_data = dataset["train"]

with open(config_path, 'r') as file:
    lora_configs = json.load(file)

models = lora_configs
model_names = []

# Compute average embeddings for each model
for model in models:
    model_name = f"Styxxxx/llama2_7b_lora-{model['model_name']}"

    model_names.append(model_name)

embeddings_list = []

# Load the embeddings array from disk
embeddings_save_path = "./test_results/test_embeddings.npy"
loaded_embeddings = np.load(embeddings_save_path)

with torch.no_grad():
    with tqdm(total=len(dataset["train"]), desc="Evaluating", unit="item") as pbar:
        for i in range(0, len(eval_data["inputs"]), batch_size):
            input_text = eval_data["inputs"][i : i + batch_size]
            task_names = eval_data["task"][i : i + batch_size]

            # If out-of-domain filtering is required, specify exclusion list
            exclude_list = None
            if ood:
                if model_size == '7b':
                    exclude_list = [f"Styxxxx/llama2_7b_lora-{task}" for task in task_names]
                else:
                    exclude_list = [f"Styxxxx/llama2_13b_lora-{task}" for task in task_names]
            
            # Perform retrieval to get top-k LoRA modules
            I_batch = loaded_embeddings[i : i + batch_size]
            I_batch = torch.tensor(I_batch, dtype=torch.bfloat16).to(device) 

            embeddings_list.append(I_batch.cpu().float().numpy())

            if ood:
                mapping_matrix_tensor, _ = scorer(I_batch, exclude_idx=model_names.index(f"Styxxxx/llama2_7b_lora-{task_names[0]}"))
            else:
                mapping_matrix_tensor, _ = scorer(I_batch)

            selected_model = torch.argmax(mapping_matrix_tensor, dim=-1).cpu().numpy()
            outputs = [model_output_dicts[selected_model[j]][input_text[j]] for j in range(len(input_text))]

            # Process and store results
            for j, (output, expected_answer) in enumerate(zip(outputs, eval_data["targets"][i : i + batch_size])):
                generated_answer = outputs[j]

                sample = {
                    'inputs': eval_data["inputs"][i+j],
                    'targets': eval_data["targets"][i+j],
                    'metric': eval_data["metric"][i+j],
                    'domain': eval_data["domain"][i+j],
                    'task': eval_data["task"][i+j],
                    'predicted_answer': generated_answer
                }
                results.append(sample)

            pbar.update(len(input_text))

# Save the results to a JSON file
os.makedirs(os.path.dirname(res_path), exist_ok=True)
with open(res_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

[info] loaded results matrix from ./performance_large/model_performance.npy with shape (48, 48)


/tmp/ipykernel_15795/1283716768.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")
Evaluating: 100%|██████████| 2395/239

In [20]:
!python3 summarize_results.py

/home/ubuntu/fall-2025-semester-project/myenv/lib/python3.10/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/ubuntu/fall-2025-semester-project/myenv/lib/python3.10/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/ubuntu/fall-2025-semester-project/myenv/lib/python3.10/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps o